# Remote Inference with NeMo Retriever

End-to-end document ingestion and retrieval-augmented generation (RAG) using **NVIDIA NIM APIs** — no local GPU required.

**Workflow:**
1. Configure environment and initialise Ray
2. Ingest a PDF via NVIDIA extraction and embedding NIMs → LanceDB
3. Run a simple single-shot RAG query
4. Explore the vector database contents
5. Run a ReAct agent that iteratively queries the knowledge base

## 1. Configuration

Set the environment variables before importing other modules. Replace `NVIDIA_API_KEY` with your key from [build.nvidia.com](https://build.nvidia.com).

In [ ]:
import os

# Scratch directory for Ray temporary files — adjust to a local path with sufficient space
os.environ["RAY_TMPDIR"] = "/raid/mwason/tmp"

# NVIDIA NIM API key — obtain from https://build.nvidia.com
os.environ["NVIDIA_API_KEY"] = "your-api-key-here"

## 2. Initialise Ray

NeMo Retriever uses Ray for distributed batch processing. The dashboard is disabled here for a lighter footprint.

In [ ]:
import ray

ray.init(include_dashboard=False)

## 3. Ingest Documents

Build the NeMo Retriever ingestion pipeline. Each stage calls a remote NVIDIA NIM:

| Stage | NIM | Purpose |
|-------|-----|---------|
| Extract | `nemotron-page-elements-v3`, `nemotron-graphic-elements-v1`, `nemoretriever-ocr-v1`, `nemotron-table-structure-v1` | Parse layout, graphics, OCR, tables |
| Embed  | `nvidia/llama-nemotron-embed-1b-v2` | Produce text embeddings |
| Upload | — | Write chunks + vectors to LanceDB |

In [ ]:
from pathlib import Path
from nemo_retriever import create_ingestor

# Update this path to point at the PDF you want to ingest
PDF_PATH = str(Path("/home/nfs/mwason/nemo/NeMo-Retriever/data/multimodal_test.pdf"))

ingestor = (
    create_ingestor(run_mode="batch")
    .files([PDF_PATH])
    .extract(
        page_elements_invoke_url="https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-page-elements-v3",
        graphic_elements_invoke_url="https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-graphic-elements-v1",
        ocr_invoke_url="https://ai.api.nvidia.com/v1/cv/nvidia/nemoretriever-ocr-v1",
        table_structure_invoke_url="https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-table-structure-v1",
    )
    .embed(
        embed_invoke_url="https://integrate.api.nvidia.com/v1/embeddings",
        model_name="nvidia/llama-nemotron-embed-1b-v2",
        embed_modality="text",
    )
    .vdb_upload()
)

### Run Ingestion

Execute the pipeline. This may take a few minutes on the first run.

In [ ]:
ray_dataset = ingestor.ingest()
chunks = ray_dataset.get_dataset().take_all()

print(f"Total chunks: {len(chunks)}")
print(f"Chunk fields: {list(chunks[0].keys())}")

### Inspect Chunk Content

Print the raw text extracted from every chunk.

In [ ]:
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ---")
    print(chunk["text"])
    print()

## 4. Simple RAG Query

Set up the `Retriever` against the LanceDB backend, then run a single-shot RAG query:
embed the question → fetch top-k chunks → pass as context to an NVIDIA LLM.

In [ ]:
from nemo_retriever.retriever import Retriever
from openai import OpenAI

retriever = Retriever(
    lancedb_uri="lancedb",
    lancedb_table="nv-ingest",
    embedder="nvidia/llama-3.2-nv-embedqa-1b-v2",
    top_k=5,
    reranker=False,
)

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ.get("NVIDIA_API_KEY"),
)

In [ ]:
query = (
    "What is the total number of tables, charts, and bullet points in the document, "
    "and which page has the most content elements?"
)

hits = retriever.query(query)
hit_texts = [hit["text"] for hit in hits]

print(f"Retrieved {len(hits)} chunks:")
for i, hit in enumerate(hits):
    print(f"\n[{i + 1}] Page {hit.get('page_number', '?')} — {hit['text'][:150]}...")

In [ ]:
prompt = f"""Given the following retrieved documents, answer the question: {query}

Documents:
{hit_texts}
"""

completion = client.chat.completions.create(
    model="nvidia/nemotron-3-super-120b-a12b",
    messages=[{"role": "user", "content": prompt}],
    stream=False,
)

print(completion.choices[0].message.content)

## 5. LanceDB Exploration

Directly inspect the LanceDB vector store — useful for debugging ingestion results, reviewing the schema, and visualising the embedding space.

### Schema & Data

In [ ]:
import lancedb

db = lancedb.connect("lancedb")
table = db.open_table("nv-ingest")

print(table.schema)

In [ ]:
df = table.to_pandas()
print(f"Columns: {list(df.columns)}\n")
print(df.head())

In [ ]:
for i, row in df.iterrows():
    print(f"--- Chunk {i} ---")
    print(f"Text:       {row['text'][:100]}")
    print(f"Page:       {row['page_number']}")
    print(f"Source:     {row['source']}")
    print(f"Vector dim: {len(row['vector'])}")
    print()

### Embedding Visualisation (PCA)

Reduce the high-dimensional embedding vectors to 2D with PCA to get an intuitive sense of how chunks cluster in the embedding space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

vectors = np.stack(df["vector"].values)
reduced = PCA(n_components=2).fit_transform(vectors)

plt.figure(figsize=(10, 6))
plt.scatter(reduced[:, 0], reduced[:, 1], alpha=0.7)
for i in range(len(df)):
    plt.annotate(f"chunk_{i}", reduced[i], fontsize=7)
plt.title("Chunk Embeddings — PCA (2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

## 6. ReAct Agent Loop

A minimal [ReAct](https://arxiv.org/abs/2210.03629) (Reason + Act) agent that iteratively searches the knowledge base before producing a final answer.

| Tool | Description |
|------|-------------|
| `query_docs` | Semantic search over the ingested vector DB |
| `ingest_pdf` | Ingest a new PDF into the knowledge base at runtime |
| `final_answer` | Return the final answer and terminate the loop |

In [ ]:
import json

AGENT_MODEL = "meta/llama-3.1-70b-instruct"
MAX_ITERATIONS = 8

In [ ]:
SYSTEM_PROMPT = """You are a research assistant with access to a document knowledge base.
Reason step by step and search the knowledge base as needed using query_docs.
Search multiple times with different queries if needed to gather enough information.
When confident, call final_answer. Do not make up information."""

### Tool Definitions

OpenAI-compatible JSON schemas for the three agent tools, followed by their Python implementations.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "query_docs",
            "description": "Search the document knowledge base for relevant text chunks. Call multiple times with different queries to gather more context.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query_string": {
                        "type": "string",
                        "description": "Search query to run against the vector database.",
                    }
                },
                "required": ["query_string"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "ingest_pdf",
            "description": "Ingest a PDF file into the knowledge base. Use only when the user provides a new PDF path.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pdf_path": {
                        "type": "string",
                        "description": "Filesystem path to the PDF.",
                    }
                },
                "required": ["pdf_path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "final_answer",
            "description": "Provide the final answer when confident. Terminates the loop.",
            "parameters": {
                "type": "object",
                "properties": {
                    "answer": {
                        "type": "string",
                        "description": "The complete answer to return.",
                    }
                },
                "required": ["answer"],
            },
        },
    },
]

In [ ]:
def run_query_docs(query_string: str) -> list[dict]:
    hits = retriever.query(query_string)
    # Drop vector field to avoid context bloat (~40 KB per query)
    return [
        {"text": h.get("text", ""), "source": h.get("pdf_basename", ""), "page": h.get("page_number", "")}
        for h in hits
    ]


def run_ingest_pdf(pdf_path: str) -> str:
    pipeline = (
        create_ingestor(run_mode="batch")
        .files([pdf_path])
        .extract(
            page_elements_invoke_url="https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-page-elements-v3",
            graphic_elements_invoke_url="https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-graphic-elements-v1",
            ocr_invoke_url="https://ai.api.nvidia.com/v1/cv/nvidia/nemoretriever-ocr-v1",
            table_structure_invoke_url="https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-table-structure-v1",
        )
        .embed(
            embed_invoke_url="https://integrate.api.nvidia.com/v1/embeddings",
            model_name="nvidia/llama-nemotron-embed-1b-v2",
            embed_modality="text",
        )
        .vdb_upload()
    )
    pipeline.ingest()
    return f"Ingested {pdf_path} successfully."

### ReAct Loop

The `react_agent` function runs the main loop, dispatching tool calls until `final_answer` is invoked or `MAX_ITERATIONS` is reached.

In [ ]:
def react_agent(question: str, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for iteration in range(1, MAX_ITERATIONS + 1):
        if verbose:
            print(f"\n[Iteration {iteration}]")

        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            parallel_tool_calls=False,
        )

        msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        if verbose:
            print(f"  finish_reason: {finish_reason}")

        messages.append(msg)

        # No tool calls — model answered directly
        if finish_reason == "stop" or not msg.tool_calls:
            return msg.content or "(No answer produced)"

        # Process tool calls
        for tool_call in msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)

            if verbose:
                print(f"  -> {fn_name}({json.dumps(fn_args)})")

            if fn_name == "final_answer":
                answer = fn_args["answer"]
                if verbose:
                    print(f"\n[Final Answer]\n{answer}")
                return answer
            elif fn_name == "query_docs":
                result = json.dumps(run_query_docs(fn_args["query_string"]), ensure_ascii=False)
            elif fn_name == "ingest_pdf":
                result = run_ingest_pdf(fn_args["pdf_path"])
            else:
                result = f"Unknown tool: {fn_name}"

            if verbose:
                print(f"     result: {result[:200]}...")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

    return f"[Agent stopped after {MAX_ITERATIONS} iterations without a final answer]"

### Run the Agent

In [ ]:
answer = react_agent(
    "What is the total number of tables, charts, and bullet points in the document, "
    "and which page has the most content elements?",
    verbose=True,
)
print("\n=== ANSWER ===")
print(answer)